# Index of Economic Freedom — G7 + China Analysis

Standalone analysis of the **Heritage Foundation Index of Economic Freedom (IEF)** data for G7 + China, covering 1995–2026.

## Data Source

Files under [`data/Index of Economic Freedom/`](../../data/Index%20of%20Economic%20Freedom/), specifically `G7+China All Data` (CSV, no extension).

## Components (12 + Overall Score)

- **Rule of Law**: Property Rights, Government Integrity, Judicial Effectiveness
- **Government Size**: Tax Burden, Government Spending, Fiscal Health
- **Regulatory Efficiency**: Business Freedom, Labor Freedom, Monetary Freedom
- **Open Markets**: Trade Freedom, Investment Freedom, Financial Freedom


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams['axes.unicode_minus'] = False

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p
    raise FileNotFoundError('Could not locate repo root containing `data/`')

repo_root = find_repo_root(Path.cwd())
ief_dir = repo_root / 'data' / 'Index of Economic Freedom'
print(f'IEF dir: {ief_dir}')
print(f'Exists:  {ief_dir.exists()}')


## 2. Load G7+China Panel

The CSV has 3 header lines (title, source, URL, blank), so we skip them.


In [ ]:
ief_path = ief_dir / 'G7+China All Data'

df = pd.read_csv(ief_path, skiprows=4, header=0)
df.columns = [c.strip() for c in df.columns]
df['Country'] = df['Country'].str.strip()

# Coerce all score columns to numeric
score_cols = [c for c in df.columns if c not in ['Country', 'Index Year']]
for c in score_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['Index Year'] = pd.to_numeric(df['Index Year'], errors='coerce').astype('Int64')

print(f'Shape: {df.shape}')
print(f'Year range: {df["Index Year"].min()} - {df["Index Year"].max()}')
print(f'Countries: {df["Country"].unique().tolist()}')
df.head()


## 3. Country Definitions

In [ ]:
COUNTRIES = ['United States', 'China', 'United Kingdom', 'France', 'Germany', 'Italy', 'Japan', 'Canada']

COLORS = {
    'United States': '#1f77b4',
    'China':         '#d62728',
    'United Kingdom':'#2ca02c',
    'France':        '#9467bd',
    'Germany':       '#8c564b',
    'Italy':         '#e377c2',
    'Japan':         '#ff7f0e',
    'Canada':        '#17becf',
}

COMPONENTS = [c for c in df.columns if c not in ['Country', 'Index Year', 'Overall Score']]
print('Components:', COMPONENTS)


## 4. Overall Score Trend (1995–2026)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for country in COUNTRIES:
    sub = df[df['Country'] == country].sort_values('Index Year')
    ax.plot(sub['Index Year'], sub['Overall Score'],
            marker='o', markersize=3, linewidth=1.8,
            label=country, color=COLORS[country])

ax.set_xlabel('Year')
ax.set_ylabel('Overall Score (0–100)')
ax.set_title('Index of Economic Freedom — Overall Score (1995–2026)',
             fontsize=13, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=True)
ax.grid(True, alpha=0.3)
ax.set_ylim(40, 90)
plt.tight_layout()
plt.show()


## 5. Latest Year (2026) Ranking

In [ ]:
latest_year = int(df['Index Year'].max())
latest = df[df['Index Year'] == latest_year].sort_values('Overall Score', ascending=True)
latest = latest[latest['Country'].isin(COUNTRIES)]

fig, ax = plt.subplots(figsize=(11, 5))
colors = [COLORS[c] for c in latest['Country']]
bars = ax.barh(latest['Country'], latest['Overall Score'], color=colors)
ax.set_xlabel('Overall Score')
ax.set_title(f'IEF Overall Score Ranking — {latest_year}', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3)
ax.axvline(70, ls='--', color='gray', alpha=0.5)
ax.axvline(60, ls='--', color='gray', alpha=0.5)
ax.text(70, -0.5, 'Mostly Free', fontsize=8, color='gray')
ax.text(60, -0.5, 'Moderately Free', fontsize=8, color='gray')

for i, (c, v) in enumerate(zip(latest['Country'], latest['Overall Score'])):
    ax.text(v + 1, i, f'{v:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()
latest[['Country', 'Overall Score']].reset_index(drop=True)


## 6. Component Heatmap (Latest Year)

In [ ]:
latest_df = df[df['Index Year'] == latest_year].set_index('Country')[COMPONENTS]
latest_df = latest_df.reindex(COUNTRIES)

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(latest_df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)

ax.set_xticks(range(len(COMPONENTS)))
ax.set_xticklabels(COMPONENTS, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(COUNTRIES)))
ax.set_yticklabels(COUNTRIES, fontsize=10)
ax.set_title(f'IEF Components by Country — {latest_year}',
             fontsize=13, fontweight='bold')

for i in range(len(COUNTRIES)):
    for j in range(len(COMPONENTS)):
        v = latest_df.values[i, j]
        if pd.notna(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                    fontsize=8, color='black' if v > 50 else 'white')

plt.colorbar(im, ax=ax, label='Score (0-100)')
plt.tight_layout()
plt.show()
latest_df.round(1)


## 7. Component Radar Profiles (Latest Year)

In [ ]:
from math import pi

N = len(COMPONENTS)
angles = [n / N * 2 * pi for n in range(N)] + [0]

fig, axes = plt.subplots(2, 4, figsize=(18, 9), subplot_kw=dict(projection='polar'))
axes = axes.flatten()

for ax, country in zip(axes, COUNTRIES):
    if country in latest_df.index:
        vals = latest_df.loc[country].tolist()
        vals += vals[:1]
        ax.plot(angles, vals, color=COLORS[country], linewidth=2)
        ax.fill(angles, vals, color=COLORS[country], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([c.replace(' ', '\n') for c in COMPONENTS], fontsize=7)
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['25', '50', '75', '100'], fontsize=7)
    ax.set_title(country, fontsize=11, fontweight='bold', pad=15)

plt.suptitle(f'IEF Component Radar — {latest_year}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Long-term Change (1995 → 2026)

In [ ]:
first_year = int(df['Index Year'].min())

start = df[df['Index Year'] == first_year].set_index('Country')['Overall Score']
end = df[df['Index Year'] == latest_year].set_index('Country')['Overall Score']
delta = (end - start).reindex(COUNTRIES).sort_values()

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in delta]
bars = ax.barh(delta.index, delta.values, color=colors)
ax.set_xlabel(f'Change in Overall Score ({first_year} → {latest_year})')
ax.set_title(f'IEF Overall Score: {first_year} vs {latest_year}',
             fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(delta.values):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f'{v:+.1f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    f'{first_year}': start.reindex(COUNTRIES).round(1),
    f'{latest_year}': end.reindex(COUNTRIES).round(1),
    'Change': (end - start).reindex(COUNTRIES).round(1),
}).sort_values('Change', ascending=False)
summary


## 9. Volatility — Which Countries Are Most Stable?

In [ ]:
stats = df[df['Country'].isin(COUNTRIES)].groupby('Country')['Overall Score'].agg(
    mean='mean', std='std', min='min', max='max'
).round(2)
stats['range'] = stats['max'] - stats['min']
stats = stats.reindex(COUNTRIES).sort_values('std')
print('Volatility ranking (lower std = more stable):')
stats


## 10. Component Trends — Small Multiples

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
axes = axes.flatten()

for ax, comp in zip(axes, COMPONENTS):
    for country in COUNTRIES:
        sub = df[df['Country'] == country].sort_values('Index Year')
        ax.plot(sub['Index Year'], sub[comp],
                color=COLORS[country], linewidth=1.2, alpha=0.85,
                label=country if comp == COMPONENTS[0] else None)
    ax.set_title(comp, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

# Shared legend
handles = [plt.Line2D([0], [0], color=COLORS[c], lw=2, label=c) for c in COUNTRIES]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.02),
           ncol=8, fontsize=9, frameon=True)

plt.suptitle('IEF Components Over Time (1995-2026)', fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()


## 11. China Deep Dive

In [ ]:
china = df[df['Country'] == 'China'].sort_values('Index Year').set_index('Index Year')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Overall Score
axes[0].plot(china.index, china['Overall Score'],
             marker='o', markersize=4, linewidth=2, color=COLORS['China'])
axes[0].fill_between(china.index, china['Overall Score'], alpha=0.15, color=COLORS['China'])
axes[0].set_ylabel('Overall Score')
axes[0].set_title('China — IEF Overall Score (1995-2026)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Component breakdown
for comp in COMPONENTS:
    axes[1].plot(china.index, china[comp], linewidth=1.2, label=comp, alpha=0.85)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Component Score')
axes[1].set_title('China — Component Trajectories', fontweight='bold')
axes[1].legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Component Correlation Within G7+China

In [ ]:
# Correlation between components, pooling all G7+China observations
panel = df[df['Country'].isin(COUNTRIES)][COMPONENTS + ['Overall Score']]
corr = panel.corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index, fontsize=9)
ax.set_title('Correlation between IEF Components (G7+China pooled, 1995-2026)',
             fontsize=12, fontweight='bold')

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        v = corr.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(v) > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
plt.show()


## 13. Save Cleaned Panel

In [ ]:
out_path = repo_root / 'data' / 'number' / 'ief_g7_china_panel.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
df_clean = df[df['Country'].isin(COUNTRIES)].sort_values(['Country', 'Index Year']).reset_index(drop=True)
df_clean.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Rows: {len(df_clean)}')
df_clean.head()
